# WZML-X — Colab deploy

This notebook runs the WZML-X Telegram mirror/leech bot on a free Colab runtime:
it clones the repo, installs the system + Python dependencies, writes your
`config.py` from the form below, and starts the bot in the background.

**Run the cells from top to bottom.** First run takes ~6–8 minutes, later runs
are faster because the clone is reused (the runtime itself resets after a
disconnect, so you re-run the whole notebook).

> Colab ke liye: `Runtime → Change runtime type → CPU` chuno (GPU ki zaroorat
> nahi), aur cells ek-ek karke chalao. Har cell ke top-left me ▶ button dabao.

**Things to know before you start**

| | |
|---|---|
| Session limit | ~12 hours, then the VM is wiped. Not meant for 24×7 hosting. |
| Disk | ~70 GB in `/content` (downloads go there). |
| RAM | ~12 GB on CPU runtime. Keep `BOT_MAX_TASKS` low. |
| Persistence | Set `DATABASE_URL` so your bot settings survive a restart. Files inside `/content` do **not** survive. |
| Public URLs | No Cloudflare tunnel here, so the web UI / stream links stay local to the VM. |

## 1. Get the code

Clones this repository (or hard-resets it if it is already there) and points
`/usr/src/app` — the path the bot has hard-coded for sessions and downloads —
at the Colab disk, so nothing fills up the small root partition.

In [ ]:
#@title 1) Clone / update the repo { display-mode: "form" }
REPO_URL = "https://github.com/gintokisakata918-prog/WZML-X.git" #@param {type:"string"}
BRANCH = "wzv3" #@param {type:"string"}

import os
import pathlib
import subprocess

APP_DIR = pathlib.Path("/content/WZML-X")


def run(cmd):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True)


APP_DIR.parent.mkdir(parents=True, exist_ok=True)
if (APP_DIR / ".git").exists():
    run(["git", "-C", str(APP_DIR), "fetch", "--depth", "1", "origin", BRANCH])
    run(["git", "-C", str(APP_DIR), "reset", "--hard", f"origin/{BRANCH}"])
else:
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(APP_DIR)])

# The bot hard-codes /usr/src/app for session files and DOWNLOAD_DIR.
link = pathlib.Path("/usr/src/app")
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink():
    if link.resolve() != APP_DIR.resolve():
        link.unlink()
elif link.exists():
    print(f"note: {link} already exists as a real directory, keeping it")
if not link.exists():
    link.symlink_to(APP_DIR, target_is_directory=True)

(APP_DIR / "downloads").mkdir(exist_ok=True)
print("\nApp dir :", APP_DIR)
print("/usr/src/app ->", os.path.realpath(link))
print("HEAD    :", subprocess.run(["git", "-C", str(APP_DIR), "log", "-1", "--oneline"],
                                   capture_output=True, text=True).stdout.strip())

## 2. System packages

`ffmpeg` (leech/thumbnails), `aria2c` + `qbittorrent-nox` (torrents/mirror),
`7z` (archive extraction), `rclone`, `mediainfo`, `cpulimit`.

Optional extras: SABnzbd for Usenet and a JRE for JDownloader. Leave both
unticked unless you actually use them — they eat RAM on a Colab VM.

In [ ]:
#@title 2) Install system packages { display-mode: "form" }
INSTALL_SABNZBD = False #@param {type:"boolean"}
INSTALL_JDOWNLOADER = False #@param {type:"boolean"}

import shutil
import subprocess

base = [
    "ffmpeg", "aria2", "qbittorrent-nox", "p7zip-full", "unzip", "rclone",
    "mediainfo", "cpulimit", "libmagic1", "curl", "git", "ca-certificates",
]
optional = []
if INSTALL_SABNZBD:
    optional.append("sabnzbdplus")
if INSTALL_JDOWNLOADER:
    optional.append("default-jre-headless")

subprocess.run(["apt-get", "-qq", "update"], check=False)
r = subprocess.run(["apt-get", "install", "-y", "-qq", "--no-install-recommends", *base],
                   check=False)
if r.returncode:
    print("\napt failed for the base set — re-running one package at a time to see which one")
    for pkg in base:
        subprocess.run(["apt-get", "install", "-y", "-qq", "--no-install-recommends", pkg],
                       check=False)
for pkg in optional:
    subprocess.run(["apt-get", "install", "-y", "-qq", "--no-install-recommends", pkg],
                   check=False)

print()
for binary in ["ffmpeg", "aria2c", "qbittorrent-nox", "7z", "rclone", "mediainfo", "cpulimit"]:
    path = shutil.which(binary)
    print(f"{binary:<18} {path or 'MISSING'}")

## 3. Python dependencies

Installs `requirements.txt`, plus two Colab-specific fixes:

- **`uv`** — `update.py` calls `uv pip install`, so it has to exist (and
  `UV_SYSTEM_PYTHON=1` makes it install into Colab's system Python).
- **`niquests>=3.15`** — the bot imports `niquests.packages.urllib3`, which only
  newer releases provide; an older resolver pick breaks the import.
- **`wz_bin.py`** — inside the official Docker image this helper module ships
  with the image. `colab/wz_bin.py` is a tiny stand-in that just resolves the
  binaries from `PATH`; it is copied into the repo root here.

In [ ]:
#@title 3) Install Python dependencies { display-mode: "form" }
import importlib
import pathlib
import shutil
import subprocess
import sys

APP_DIR = pathlib.Path("/content/WZML-X")


def sh(cmd, cwd=APP_DIR):
    print("+", cmd)
    result = subprocess.run(cmd, shell=True, cwd=str(cwd))
    if result.returncode:
        print(f"  (exit {result.returncode})")
    return result.returncode


sh(f"{sys.executable} -m pip install -q --upgrade pip uv")
sh(f"{sys.executable} -m pip install -q -r requirements.txt")
sh(f'{sys.executable} -m pip install -q -U "niquests>=3.15"')

# stand-in for the helper that lives inside the docker image
shutil.copy(APP_DIR / "colab" / "wz_bin.py", APP_DIR / "wz_bin.py")

# smoke test: the bot package must import cleanly before we try to start it
check = subprocess.run(
    [sys.executable, "-c",
     "import bot, bot.core.config_manager as c; "
     "print('imports OK | aria2 =', c.BinConfig.ARIA2_NAME, "
     "'| qbit =', c.BinConfig.QBIT_NAME, '| ffmpeg =', c.BinConfig.FFMPEG_NAME)"],
    cwd=str(APP_DIR), capture_output=True, text=True,
)
print(check.stdout.strip() or check.stderr.strip()[-2000:])

## 4. Config

Fill the form and run the cell — it writes `config.py` into the repo, keeping
every default from `config_sample.py` that you do not touch.

| Field | Where to get it |
|---|---|
| `BOT_TOKEN` | [@BotFather](https://t.me/BotFather) → `/newbot` |
| `TELEGRAM_API`, `TELEGRAM_HASH` | [my.telegram.org](https://my.telegram.org) → API development tools |
| `OWNER_ID` | your numeric Telegram id ([@userinfobot](https://t.me/userinfobot)) |
| `DATABASE_URL` | MongoDB URI (Atlas free tier is fine). Strongly recommended — bot settings live here. |
| `USER_SESSION_STRING` | optional: Pyrogram v2 session string, needed for 4 GB uploads / user-only links |
| `BASE_URL` | optional: leave empty on Colab (no public tunnel) |

Use **Extra config** for anything not in the form, one `KEY = value` per line,
e.g.

```
LEECH_SPLIT_SIZE = 2097152000
DISABLE_TORRENTS = False
CMD_SUFFIX = "x"
```

In [ ]:
#@title 4) Write config.py { display-mode: "form" }
#@markdown Required: BOT_TOKEN, OWNER_ID, TELEGRAM_API, TELEGRAM_HASH.
BOT_TOKEN = "" #@param {type:"string"}
OWNER_ID = 0 #@param {type:"integer"}
TELEGRAM_API = 0 #@param {type:"integer"}
TELEGRAM_HASH = "" #@param {type:"string"}
DATABASE_URL = "" #@param {type:"string"}
USER_SESSION_STRING = "" #@param {type:"string"}
BASE_URL = "" #@param {type:"string"}
DISABLE_JD = True #@param {type:"boolean"}
DISABLE_NZB = True #@param {type:"boolean"}
EXTRA_CONFIG = "" #@param {type:"string"}

import pathlib
import re

APP_DIR = pathlib.Path("/content/WZML-X")

missing = [name for name, value in (
    ("BOT_TOKEN", BOT_TOKEN), ("OWNER_ID", OWNER_ID),
    ("TELEGRAM_API", TELEGRAM_API), ("TELEGRAM_HASH", TELEGRAM_HASH),
) if not value]
assert not missing, f"Fill these in the form first: {', '.join(missing)}"
assert re.fullmatch(r"\d+:[A-Za-z0-9_-]{20,}", BOT_TOKEN), "BOT_TOKEN does not look like a bot token"

# start from the sample so every untouched default stays exactly as shipped
namespace = {}
exec((APP_DIR / "config_sample.py").read_text(), namespace)

overrides = {
    "BOT_TOKEN": BOT_TOKEN.strip(),
    "OWNER_ID": int(OWNER_ID),
    "TELEGRAM_API": int(TELEGRAM_API),
    "TELEGRAM_HASH": TELEGRAM_HASH.strip(),
    "DATABASE_URL": DATABASE_URL.strip(),
    "USER_SESSION_STRING": USER_SESSION_STRING.strip(),
    "BASE_URL": BASE_URL.strip().rstrip("/"),
    "DISABLE_JD": bool(DISABLE_JD),
    "DISABLE_NZB": bool(DISABLE_NZB),
    "TIMEZONE": "Asia/Kolkata",
    # Colab has no public tunnel, so keep the web UI/stream to localhost only
    "ENABLE_TELEMETRY": True,
}
namespace.update(overrides)

# "KEY = value" lines from the Extra config box
for line in EXTRA_CONFIG.splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    match = re.fullmatch(r"([A-Z][A-Z0-9_]*)\s*=\s*(.+)", line)
    if not match:
        print(f"skipped (not a KEY = value line): {line}")
        continue
    key, raw = match.groups()
    try:
        namespace[key] = eval(raw, {}, {})  # noqa: S307 - your own notebook input
    except Exception:
        namespace[key] = raw.strip().strip("\"'")

keys = [key for key in namespace if key.isupper()]
body = ["# Generated by colab.ipynb - change values in the notebook form, not here.", ""]
body += [f"{key} = {namespace[key]!r}" for key in keys]
(APP_DIR / "config.py").write_text("\n".join(body) + "\n")

print(f"config.py written ({len(keys)} keys) -> {APP_DIR / 'config.py'}")
for key in ("BOT_TOKEN", "OWNER_ID", "TELEGRAM_API", "DATABASE_URL", "DISABLE_JD", "DISABLE_NZB"):
    value = namespace[key]
    shown = f"{value[:6]}...{value[-4:]}" if key == "BOT_TOKEN" and value else value
    print(f"  {key:<16} {shown}")

## 5. Start the bot

`start.sh` runs `update.py` (checks config/ENVs, refreshes packages) and then
`python -m bot`. It runs in the background so the cell returns — the first
~40 seconds are dependency refresh, then the bot logs in and registers its
commands.

> `UPSTREAM_REPO` ko Colab me **khaali chhodo**. Set karoge to `update.py` aapka
> `.git` delete karke hard reset kar dega, aur `colab.ipynb` bhi gayab ho jayega.

In [ ]:
#@title 5) Start the bot { display-mode: "form" }
import os
import pathlib
import signal
import subprocess
import time

APP_DIR = pathlib.Path("/content/WZML-X")
LOG = pathlib.Path("/content/wzmlx.log")
PID_FILE = pathlib.Path("/content/wzmlx.pid")


def stop_running():
    if PID_FILE.exists():
        try:
            os.killpg(os.getpgid(int(PID_FILE.read_text())), signal.SIGTERM)
            print("stopped previous run")
        except (ProcessLookupError, ValueError):
            pass
        PID_FILE.unlink(missing_ok=True)
    subprocess.run("pkill -f 'python -m bot'; pkill -f 'qbittorrent-nox'; "
                   "pkill -f 'aria2c'; pkill -f 'gunicorn'",
                   shell=True, check=False)


stop_running()

env = os.environ.copy()
env.update({
    "UV_SYSTEM_PYTHON": "1",   # update.py calls `uv pip install`
    "PYTHONUNBUFFERED": "1",
    "PORT": "8080",            # local web UI / cron hooks
})
with open(LOG, "wb") as logfile:
    process = subprocess.Popen(
        ["bash", "start.sh"],
        cwd=str(APP_DIR), env=env,
        stdout=logfile, stderr=subprocess.STDOUT,
        start_new_session=True,
    )
PID_FILE.write_text(str(process.pid))
print(f"started pid {process.pid}, log -> {LOG}")
print("waiting 40s for install/update + login...\n")
time.sleep(40)
print(LOG.read_text(errors="replace")[-4000:])

## 6. Logs & control

In [ ]:
#@title 6) Tail the log { display-mode: "form" }
LINES = 60 #@param {type:"integer"}

import pathlib
import subprocess

LOG = pathlib.Path("/content/wzmlx.log")
PID_FILE = pathlib.Path("/content/wzmlx.pid")

alive = "no"
if PID_FILE.exists():
    pid = PID_FILE.read_text().strip()
    alive = subprocess.run(["bash", "-c", f"kill -0 {pid} 2>/dev/null && echo yes || echo no"],
                           capture_output=True, text=True).stdout.strip()
print(f"bot running: {alive}\n")
print(subprocess.run(["tail", "-n", str(LINES), str(LOG)],
                     capture_output=True, text=True).stdout)

In [ ]:
#@title 7) Stop the bot { display-mode: "form" }
import os
import pathlib
import signal
import subprocess

PID_FILE = pathlib.Path("/content/wzmlx.pid")
if PID_FILE.exists():
    try:
        os.killpg(os.getpgid(int(PID_FILE.read_text())), signal.SIGTERM)
    except (ProcessLookupError, ValueError):
        pass
    PID_FILE.unlink(missing_ok=True)
subprocess.run("pkill -f 'python -m bot'; pkill -f 'qbittorrent-nox'; "
               "pkill -f 'aria2c'; pkill -f 'gunicorn'", shell=True, check=False)
print("stopped. Re-run cell 5 to start again.")

## Troubleshooting

| Symptom | Fix |
|---|---|
| `MISSING` next to a binary in cell 2 | Re-run cell 2; if `qbittorrent-nox` still fails, set `DISABLE_TORRENTS = True` in Extra config (cell 4) and mirror via direct links only. |
| `ModuleNotFoundError: niquests.packages` | Re-run cell 3 — it pins `niquests>=3.15`. |
| `ModuleNotFoundError: wz_bin` | Cell 3 did not finish; it copies `colab/wz_bin.py` to the repo root. |
| `BOT_TOKEN variable is missing!` | Cell 4 did not run after cloning, or `config.py` was wiped by a repo reset — re-run cell 4. |
| `Database ERROR` / settings not saving | `DATABASE_URL` is empty or unreachable. Atlas free tier: allow `0.0.0.0/0` in Network Access. |
| Bot dies after some hours | Colab disconnects idle/free runtimes (~12 h cap). Nothing to fix in the bot; use a VPS for always-on. |
| Runtime says "out of disk" | Downloads are in `/content/WZML-X/downloads`; clear them or `Runtime → Restart`. |
| `colab.ipynb` disappeared | You set `UPSTREAM_REPO`; `update.py` reset the repo. Leave it empty and re-run cell 1. |

Notes

- The Colab build is for testing and light use. For 24×7 mirroring, the Docker
  path in the README is the supported route.
- `colab/wz_bin.py` only exists to stand in for the helper module baked into the
  Docker image; Docker deployments never load it.